In [1]:
import os
import json
import pandas as pd
import winsound
import time
import gc
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# --- CONFIG PATH ---
PATH_USGS = r"D:\indonesia_output\katalog_usgs_master_2001_2025.csv"
PATH_BMKG = r"D:\indonesia_output\Indonesian_Earthquake_Catalog_BMKG_1998_2024\BMKG_Earthquake_Catalog.csv"
OUTPUT_CSV = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"

# --- ACADEMIC IDENTITY ---
ACADEMIC_USER_AGENT = (
    "ResearchProject: Doctoral Dissertation in AI and Edge Computing; "
    "Researcher: Very Kurnia Bakti (Indonesia); "
    "Contact: verykurniabakti@gmail.com, verykurniabakti@harkatnegeri.ac.id; "
    "ID: Scopus:57209452703, Scholar:9-5-_dlAAAAJ, WOS:GPG-0207-2022"
)

STATION_CODES = ["BJI", "BBJI", "GSI", "KAPI", "LBMI", "LUWI", "PMBI", "PPI", "SANI", 
                 "TOLI", "TNTI", "JAGI", "PLAI", "UGM", "BOAB", "FAU", "MSI", "BKB"]

RC_BORDERS = [
    {'code': 'RC_01', 'lat': [0.5, 6.0], 'lon': [92.0, 109.0]},
    {'code': 'RC_02', 'lat': [3.0, 14.0], 'lon': [92.0, 109.0]},
    {'code': 'RC_03', 'lat': [7.0, 14.0], 'lon': [113.5, 122.5]},
    {'code': 'RC_04', 'lat': [0.0, 7.5], 'lon': [113.5, 124.0]},
    {'code': 'RC_05', 'lat': [6.0, 7.5], 'lon': [132.0, 141.0]},
    {'code': 'RC_06', 'lat': [1.0, 3.5], 'lon': [92.0, 109.0]},
    {'code': 'RC_07', 'lat': [0.0, 14.0], 'lon': [108.5, 114.0]},
    {'code': 'RC_08', 'lat': [7.0, 14.0], 'lon': [122.0, 141.0]},
    {'code': 'RC_09', 'lat': [0.0, 7.5], 'lon': [123.5, 132.5]},
    {'code': 'RC_10', 'lat': [0.5, 6.0], 'lon': [108.5, 132.5]},
]

clients = [
    ("IRIS", Client("EARTHSCOPE", timeout=25, user_agent=ACADEMIC_USER_AGENT)),
    ("GFZ", Client("GFZ", timeout=25, user_agent=ACADEMIC_USER_AGENT))
]

def get_rc_code(lat, lon):
    for rc in RC_BORDERS:
        if rc['lat'][0] <= lat <= rc['lat'][1] and rc['lon'][0] <= lon <= rc['lon'][1]:
            return rc['code']
    return "OUT_OF_RC"

def audit_event(event_data):
    eid, timestamp, lat, lon, mag, catalog, rc_code = event_data
    found_in_event = []
    
    # Anti-throttling micro-delay to pace requests nicely
    time.sleep(0.1)
    
    try:
        t_event = UTCDateTime(pd.to_datetime(timestamp).isoformat())
        for c_name, cl in clients:
            try:
                inv = cl.get_stations(starttime=t_event-5, endtime=t_event+5, 
                                      network="*", station=",".join(STATION_CODES), level="channel")
                for net in inv:
                    for sta in net:
                        chans = [c.code for c in sta]
                        bb = [c for c in chans if c[:2] in ['BH','HH','EH']]
                        if len(set(bb)) >= 3:
                            found_in_event.append({
                                "Event_ID": eid, "Time_UTC": str(t_event), "RC": rc_code,
                                "Station": sta.code, "Net": net.code, "Lat_Sta": sta.latitude,
                                "Lon_Sta": sta.longitude, "Mag": mag, "Catalog": catalog, "Server": c_name
                            })
                del inv
            except: 
                continue
    except: 
        pass
        
    # Free thread memory instantly
    gc.collect()
    return found_in_event

def run_academic_audit():
    print(f"🚀 Starting Stabilized Audit Pipeline (RAM-Safe & Anti-Throttling Mode)...")
    
    # 1. Load & Process Catalogs Safely
    df_u = pd.read_csv(PATH_USGS).rename(columns={'time':'Timestamp', 'latitude':'lat', 'longitude':'lon', 'mag':'Mag', 'id':'ID'})
    df_u['Src'] = 'USGS'
    
    df_b = pd.read_csv(PATH_BMKG).rename(columns={'Latitude':'lat', 'Longitude':'lon', 'Magnitude':'Mag', 'Event ID':'ID'})
    df_b['Timestamp'] = pd.to_datetime(df_b['Date'].astype(str) + ' ' + df_b['Time (UTC)'].astype(str))
    df_b['Src'] = 'BMKG'
    
    df_all = pd.concat([df_u[['Timestamp', 'lat', 'lon', 'Mag', 'ID', 'Src']], 
                        df_b[['Timestamp', 'lat', 'lon', 'Mag', 'ID', 'Src']]], ignore_index=True).drop_duplicates(subset=['ID'])
    
    del df_u, df_b
    gc.collect()
    
    print("🌐 Calculating Regional Coordinate Boundaries (RC)...")
    df_all['RC'] = df_all.apply(lambda x: get_rc_code(x['lat'], x['lon']), axis=1)
    df_task = df_all[df_all['RC'] != "OUT_OF_RC"].copy()
    
    del df_all
    gc.collect()
    
    # 2. Optimized Chunked Resume Logic
    processed_ids = set()
    total_records_in_file = 0
    if os.path.exists(OUTPUT_CSV):
        # Only parse the single verification column into RAM
        df_old = pd.read_csv(OUTPUT_CSV, usecols=['Event_ID'])
        processed_ids = set(df_old['Event_ID'].astype(str).unique())
        total_records_in_file = len(df_old)
        print(f"🔄 Resuming Progress: {len(processed_ids)} events ({total_records_in_file} lines) already committed to Disk.")
        del df_old
        gc.collect()

    tasks = []
    for _, row in df_task.iterrows():
        if str(row['ID']) not in processed_ids:
            tasks.append((row['ID'], row['Timestamp'], row['lat'], row['lon'], row['Mag'], row['Src'], row['RC']))

    del df_task
    gc.collect()

    # 3. Streamed Processing Loop
    temp_list = []
    last_beep = time.time()
    
    print(f"📡 Launching audit pool across {len(tasks)} target events...")

    # Max workers kept at 4 to optimize Core-for-Core execution on Intel i5-3570
    with ThreadPoolExecutor(max_workers=4) as executor:
        pbar = tqdm(total=len(tasks), unit="event", desc="Audit Progress")
        
        for result in executor.map(audit_event, tasks):
            if result:
                temp_list.extend(result)
                total_records_in_file += len(result)
                
                # Low memory threshold flush (every 100 entries)
                if len(temp_list) >= 100:
                    df_to_append = pd.DataFrame(temp_list)
                    file_exists = os.path.isfile(OUTPUT_CSV)
                    df_to_append.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists)
                    
                    # Complete heap cleanup
                    temp_list = []
                    del df_to_append
                    gc.collect()

            if time.time() - last_beep > 10:
                winsound.Beep(800, 100)
                last_beep = time.time()
            
            pbar.update(1)
            pbar.set_postfix({"Total_Rows": total_records_in_file})

    # Flush remaining cache
    if temp_list:
        pd.DataFrame(temp_list).to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.isfile(OUTPUT_CSV))

    print(f"✅ Extraction Successful! Master file safe at: {OUTPUT_CSV}")
    winsound.Beep(1500, 500)

if __name__ == "__main__":
    run_academic_audit()

🚀 Starting Stabilized Audit Pipeline (RAM-Safe & Anti-Throttling Mode)...
🌐 Calculating Regional Coordinate Boundaries (RC)...
🔄 Resuming Progress: 102466 events (1812487 lines) already committed to Disk.
📡 Launching audit pool across 7211 target events...


Audit Progress: 100%|██████████| 7211/7211 [5:17:34<00:00,  2.38s/event, Total_Rows=1968668]  

✅ Extraction Successful! Master file safe at: D:\indonesia_output\MASTER_STATION_INVENTORY.csv


Audit Progress: 100%|██████████| 7211/7211 [5:17:34<00:00,  2.64s/event, Total_Rows=1968668]


In [2]:
import pandas as pd
import numpy as np
import os
import gc

CSV_PATH = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"

def investigasi_master_data():
    print("⏳ Membuka dan menganalisis 1,96 Juta baris data... Mohon tunggu.")
    
    if not os.path.exists(CSV_PATH):
        print(f"❌ File tidak ditemukan di {CSV_PATH}")
        return

    # Baca data
    df = pd.read_csv(CSV_PATH)
    total_rows = len(df)
    
    print("\n" + "="*50)
    print("📊 LAPORAN INVESTIGASI AWAL BIG DATA SEISMOLOGI")
    print("="*50)
    print(f"🔹 Total Rekaman (Baris)       : {total_rows:,}")
    print(f"🔹 Ukuran File CSV             : {os.path.getsize(CSV_PATH) / (1024*1024):.2f} MB")
    
    # 1. Cek Missing Values
    print("\n🔍 [1] Pengecekan Data Kosong (Null/NaN):")
    missing = df.isnull().sum()
    for col, val in missing.items():
        print(f"   - Kolom {col.ljust(12)}: {val} data kosong")

    # 2. Distribusi Data Berdasarkan Stasiun Seismik
    print("\n📡 [2] Kontribusi Data per Stasiun Seismik (Top 20):")
    station_counts = df['Station'].value_counts()
    for sta, count in station_counts.items():
        percentage = (count / total_rows) * 100
        print(f"   - Stasiun {sta.ljust(6)}: {count:,} rekaman ({percentage:.2f}%)")

    # 3. Distribusi Berdasarkan Server/Node FDSN
    print("\n🌐 [3] Distribusi Sumber Data Server:")
    server_counts = df['Server'].value_counts()
    for srv, count in server_counts.items():
        percentage = (count / total_rows) * 100
        print(f"   - Server {srv.ljust(6)} : {count:,} rekaman ({percentage:.2f}%)")

    # 4. Distribusi Spasial Berdasarkan Kode Wilayah (RC)
    print("\n🗺️ [4] Kepadatan Seismisitas per Wilayah (Cluster RC):")
    rc_counts = df['RC'].value_counts()
    for rc, count in rc_counts.items():
        print(f"   - Wilayah {rc.ljust(6)}: {count:,} rekaman")

    # 5. Cek Nilai Unik Event Gempa
    unique_events = df['Event_ID'].nunique()
    print("\n🎯 [5] Statistik Event Kombinasional:")
    print(f"   - Total Event Unik yang Lolos Validasi: {unique_events:,} Gempa")
    print(f"   - Rata-rata Stasiun Terkunci per Event: {total_rows / unique_events:.1f} Stasiun")

    # Bersihkan memori
    del df
    gc.collect()

if __name__ == "__main__":
    investigasi_master_data()

⏳ Membuka dan menganalisis 1,96 Juta baris data... Mohon tunggu.

📊 LAPORAN INVESTIGASI AWAL BIG DATA SEISMOLOGI
🔹 Total Rekaman (Baris)       : 1,968,668
🔹 Ukuran File CSV             : 168.78 MB

🔍 [1] Pengecekan Data Kosong (Null/NaN):
   - Kolom Event_ID    : 0 data kosong
   - Kolom Time_UTC    : 0 data kosong
   - Kolom RC          : 0 data kosong
   - Kolom Station     : 0 data kosong
   - Kolom Net         : 0 data kosong
   - Kolom Lat_Sta     : 0 data kosong
   - Kolom Lon_Sta     : 0 data kosong
   - Kolom Mag         : 0 data kosong
   - Kolom Catalog     : 0 data kosong
   - Kolom Server      : 0 data kosong

📡 [2] Kontribusi Data per Stasiun Seismik (Top 20):
   - Stasiun BOAB  : 323,464 rekaman (16.43%)
   - Stasiun UGM   : 212,602 rekaman (10.80%)
   - Stasiun GSI   : 185,678 rekaman (9.43%)
   - Stasiun TNTI  : 174,465 rekaman (8.86%)
   - Stasiun PMBI  : 169,272 rekaman (8.60%)
   - Stasiun LUWI  : 163,702 rekaman (8.32%)
   - Stasiun JAGI  : 160,853 rekaman (8.17%)
 

In [3]:
import pandas as pd

INPUT_PATH = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"
CLEAN_CSV_PATH = r"D:\indonesia_output\INDONESIA_STATION_INVENTORY_CLEAN.csv"

print("⏳ Membersihkan dataset dari stasiun luar negeri...")
df = pd.read_csv(INPUT_PATH)

# Membuang stasiun BOAB
df_clean = df[df['Station'] != 'BOAB'].copy()

# Memperbaiki penamaan label RC yang kosmetik (jika ada string 'RC' saja, diubah ke RC_01 atau disesuaikan)
df_clean['RC'] = df_clean['RC'].replace('RC', 'RC_01_FIXED')

print(f"🔹 Baris sebelum dibersihkan: {len(df):,}")
print(f"🔹 Baris setelah dibersihkan : {len(df_clean):,}")
print(f"🔹 Jumlah baris BOAB yang dibuang: {len(df) - len(df_clean):,}")

# Simpan file bersih
df_clean.to_csv(CLEAN_CSV_PATH, index=False)
print(f"✅ Selesai! File siap dipindahkan ke MacBook: {CLEAN_CSV_PATH}")

⏳ Membersihkan dataset dari stasiun luar negeri...
🔹 Baris sebelum dibersihkan: 1,968,668
🔹 Baris setelah dibersihkan : 1,645,204
🔹 Jumlah baris BOAB yang dibuang: 323,464
✅ Selesai! File siap dipindahkan ke MacBook: D:\indonesia_output\INDONESIA_STATION_INVENTORY_CLEAN.csv
